In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV


In [14]:
train = pd.read_csv('/Users/s932172@aics.espritscholen.nl/Desktop/game development/deep_learning/lesson_5/train_black_friday.csv')
test = pd.read_csv('/Users/s932172@aics.espritscholen.nl/Desktop/game development/deep_learning/lesson_5/test_black_friday.csv')
print(train['Product_ID'])

0         P00069042
1         P00248942
2         P00087842
3         P00085442
4         P00285442
            ...    
550063    P00372445
550064    P00375436
550065    P00375436
550066    P00375436
550067    P00371644
Name: Product_ID, Length: 550068, dtype: object


In [15]:
train['Product_ID'] = train['Product_ID'].astype('category').cat.codes
test['Product_ID'] = test['Product_ID'].astype('category').cat.codes
print(train['Product_ID'])

0          672
1         2376
2          852
3          828
4         2734
          ... 
550063    3567
550064    3568
550065    3568
550066    3568
550067    3566
Name: Product_ID, Length: 550068, dtype: int16


In [16]:
train['Type'] = 'Train'
test['Type'] = 'Test'
fullData = pd.concat([train, test], axis=0)

In [17]:
ID_col = ['User_ID', 'Product_ID']
flag_col = ['Type']
target_col = ["Purchase"]
cat_cols = ['Gender', 'Age', 'City_Category', 'Stay_In_Current_City_Years']
num_cols = list(
    set(fullData.columns)
    - set(cat_cols)
    - set(ID_col)
    - set(target_col)
    - set(flag_col)
    )


In [18]:
for var in cat_cols + num_cols:
    if fullData[var].isnull().any():
        fullData[var + '_NA'] = fullData[var].isnull() * 1

fullData[num_cols] = fullData[num_cols].fillna(fullData[num_cols].mean())
fullData[cat_cols] = fullData[cat_cols].fillna(-9999)


In [19]:
for var in cat_cols:
    le = LabelEncoder()
    fullData[var] = le.fit_transform(fullData[var].astype('str'))

features = list(
    set(fullData.columns)
    - set(ID_col)
    - set(target_col)
    - set(flag_col)
    )
fullData[features] = fullData[features] / fullData[features].max()

In [20]:
train_data = fullData[fullData['Type'] == 'Train']
test_data = fullData[fullData['Type'] == 'Test']

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(train_data[features].values)

y = train_data[target_col].values

x_train, x_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)

In [21]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(x_train, y_train.ravel())

rf_preds = rf.predict(x_valid)

rf_rmse = np.sqrt(mean_squared_error(y_valid, rf_preds))
rf_mae = mean_absolute_error(y_valid, rf_preds)
print(f"Random Forest MAE: {rf_mae}")
print(f"Random Forest RMSE: {rf_rmse}")

Random Forest MAE: 2228.3864744645675
Random Forest RMSE: 3063.3856504109995


In [22]:
model = Sequential([
    Input(shape=(len(features),)),
    Dense(200, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.1),
    Dense(100, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.1),
    Dense(50, activation="relu"),
    Dense(1)
])

model.compile(loss="mean_squared_error", optimizer=Adam(learning_rate=0.001), metrics=["mean_squared_error"])

In [23]:
early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)

history = model.fit(x_train, y_train,
                    epochs=10,
                    validation_data=(x_valid, y_valid),
                    callbacks=[early_stopping],
                    verbose=1)

Epoch 1/10
12033/12033 ━━━━━━━━━━━━━━━━━━━━ 10s 697us/step - loss: 24785756.0000 - mean_squared_error: 24785742.0000 - val_loss: 17752944.0000 - val_mean_squared_error: 17752928.0000
Epoch 2/10
12033/12033 ━━━━━━━━━━━━━━━━━━━━ 8s 659us/step - loss: 17527576.0000 - mean_squared_error: 17527556.0000 - val_loss: 14818263.0000 - val_mean_squared_error: 14818234.0000
Epoch 3/10
12033/12033 ━━━━━━━━━━━━━━━━━━━━ 8s 663us/step - loss: 15004444.0000 - mean_squared_error: 15004408.0000 - val_loss: 12910495.0000 - val_mean_squared_error: 12910451.0000
Epoch 4/10
12033/12033 ━━━━━━━━━━━━━━━━━━━━ 8s 655us/step - loss: 13342491.0000 - mean_squared_error: 13342448.0000 - val_loss: 11686156.0000 - val_mean_squared_error: 11686110.0000
Epoch 5/10
12033/12033 ━━━━━━━━━━━━━━━━━━━━ 8s 652us/step - loss: 12573722.0000 - mean_squared_error: 12573677.0000 - val_loss: 11033282.0000 - val_mean_squared_error: 11033226.0000
Epoch 6/10
12033/12033 ━━━━━━━━━━━━━━━━━━━━ 8s 655us/step - loss: 12157713.0000 - mean_sq

In [24]:
nn_preds = model.predict(x_valid)
nn_mae = mean_absolute_error(y_valid, nn_preds)
print(f"Neural Network MAE: {nn_mae}")
nn_rmse = np.sqrt(mean_squared_error(y_valid, nn_preds))
print(f"Neural Network RMSE: {nn_rmse}")

final_preds = (rf_preds + nn_preds.flatten()) / 2
ensemble_mae = mean_absolute_error(y_valid, final_preds)
print(f"Ensemble MAE: {ensemble_mae}")
ensemble_rmse = np.sqrt(mean_squared_error(y_valid, final_preds))
print(f"Ensemble RMSE: {ensemble_rmse}")

5157/5157 ━━━━━━━━━━━━━━━━━━━━ 1s 236us/step
Neural Network MAE: 2363.26799084676
Neural Network RMSE: 3121.8339964028237
Ensemble MAE: 2211.424487192384
Ensemble RMSE: 2959.5717000512636
